In [2]:
import pandas as pd
import os
import yaml
import sys
import json
import random
import hashlib
from itertools import product
import torch as t
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
import outlines
from outlines import Generator
from openai import OpenAI
from datasets import Dataset, load_dataset
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
sys.path.append("../")
from src.utils import list_to_str, openai_api_call, openai_answer
device = "cpu"

In [3]:
class OpenaiResponse(BaseModel):
    response: str

def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [4]:
synthetic_sjts = read_json('sjt_data/synthetic_generated_sjt_list.json')

In [5]:
# Login using e.g. `huggingface-cli login` to access this dataset
hf_persona_dataset = load_dataset("thoughtworks/psychometric_personas")
hf_persona_dataset = hf_persona_dataset['train']

In [6]:
hf_persona_dataset['persona_text'][0]

"Demographic Fields: Name: Leonardo Cortez (Leo)\nDemographic Fields: Age: 35\nDemographic Fields: Location: Phoenix, AZ\nBehavioral and Psychological Descriptors: Appearance: Leonardo maintains a clean-shaven look with neatly combed black hair. His attire is professional, favoring crisp blue shirts and tailored suits that complement his athletic frame. Sharp brown eyes, accompanied by a confident stance, reflect years of disciplined service. Occasionally, a hint of tension shows in his posture.\nBehavioral and Psychological Descriptors: Behavior: Leo exudes an assertive energy that makes him stand out in a crowd. He communicates effectively, gesturing modestly to emphasize points. He establishes quick rapport, putting others at ease while exhibiting an inquisitive nature. Responsive to cues, his interactions are focused and sincere.\nBehavioral and Psychological Descriptors: Mood/Affect: Demonstrates a stable mood with a calm demeanor. Displays controlled responses that accommodate th

In [7]:
print(hf_persona_dataset['persona_text'][0].split('script_version')[0])

Demographic Fields: Name: Leonardo Cortez (Leo)
Demographic Fields: Age: 35
Demographic Fields: Location: Phoenix, AZ
Behavioral and Psychological Descriptors: Appearance: Leonardo maintains a clean-shaven look with neatly combed black hair. His attire is professional, favoring crisp blue shirts and tailored suits that complement his athletic frame. Sharp brown eyes, accompanied by a confident stance, reflect years of disciplined service. Occasionally, a hint of tension shows in his posture.
Behavioral and Psychological Descriptors: Behavior: Leo exudes an assertive energy that makes him stand out in a crowd. He communicates effectively, gesturing modestly to emphasize points. He establishes quick rapport, putting others at ease while exhibiting an inquisitive nature. Responsive to cues, his interactions are focused and sincere.
Behavioral and Psychological Descriptors: Mood/Affect: Demonstrates a stable mood with a calm demeanor. Displays controlled responses that accommodate the situ

In [8]:
synthetic_sjts[0].keys()

dict_keys(['base_scenario', 'config', 'hash_id', 'question', 'honesty_humility_option', 'emotionality_option', 'extraversion_option', 'agreeableness_option', 'conscientiousness_option', 'openness_option'])

In [9]:
persona_sjt_template = outlines.Template.from_string("""
<|im_start>user
You are a law enforcement officer with following attributes :

{{attributes}}

Task: Answer the below multiple choice questions:

Question: {{ question }}

Choices for these questions are:

{{ answer_options }}

Answer the question using one of the values from [1,2,3,4,5,6].
<|im_end>
<|im_start>assistant
""")

In [10]:
[synthetic_sjts[0][key] for key in synthetic_sjts[0].keys() if "_option" in key]

['Insist on full factual disclosure: separate the man from others for safety, document your objective observations and actions in detail including the supervisor’s instruction to shorten the report, notify the duty sergeant that you will submit a complete account, and arrange follow-up support resources for the subject.',
 'Prioritize de-escalation and emotional management: focus on calming the man and reducing tension with the neighbor, request immediate backup or a specialist to help stabilize the scene, take brief on-scene notes to address the urgent disturbance, and plan to complete a more thorough written account once you have composed yourself.',
 'Take charge of the scene: assign nearby personnel specific tasks, direct the man to a safer location away from doors while coordinating others to keep bystanders back, state that you will handle the paperwork after the safety actions are taken, and record a concise on-scene log before closing the call.',
 'Keep the peace and follow the

In [11]:
print(persona_sjt_template(question = synthetic_sjts[0]['question'],
                        attributes = hf_persona_dataset['persona_text'][0].split('script_version')[0],
                        answer_options = list_to_str([synthetic_sjts[0][key] for key in synthetic_sjts[0].keys() if "_option" in key])))

<|im_start>user
You are a law enforcement officer with following attributes :

Demographic Fields: Name: Leonardo Cortez (Leo)
Demographic Fields: Age: 35
Demographic Fields: Location: Phoenix, AZ
Behavioral and Psychological Descriptors: Appearance: Leonardo maintains a clean-shaven look with neatly combed black hair. His attire is professional, favoring crisp blue shirts and tailored suits that complement his athletic frame. Sharp brown eyes, accompanied by a confident stance, reflect years of disciplined service. Occasionally, a hint of tension shows in his posture.
Behavioral and Psychological Descriptors: Behavior: Leo exudes an assertive energy that makes him stand out in a crowd. He communicates effectively, gesturing modestly to emphasize points. He establishes quick rapport, putting others at ease while exhibiting an inquisitive nature. Responsive to cues, his interactions are focused and sincere.
Behavioral and Psychological Descriptors: Mood/Affect: Demonstrates a stable moo

In [12]:
openai_model_name = "gpt-4.1-mini"
openai_model = outlines.from_openai(OpenAI(), openai_model_name)

In [24]:
answer_shuffle = False
answer_index = [0,1,2,3,4,5]
if answer_shuffle:
    random.shuffle(answer_index)
    
sjt_answers = []
for persona_dataset in [hf_persona_dataset[0]]:
    persona = persona_dataset['persona_text']
    persona_answers = []
    question_hashes = []
    for sjt in synthetic_sjts[:2]:
        answer_options = [sjt[key] for key in sjt.keys() if "_option" in key]

        answer_options = [answer_options[idx] for idx in answer_index]
        question = sjt['question']
        
        prompt = persona_sjt_template(question = question,
                        attributes = persona.split('script_version')[0],
                        answer_options = list_to_str(answer_options))

        response = openai_model(prompt,OpenaiResponse)
        persona_answers.append(json.loads(response)['response'])
        question_hashes.append(sjt['hash_id'])
    persona_dict = {}
    persona_dict['config'] = {}
    persona_dict['config']['persona_id'] = persona_dataset['uuid']
    persona_dict['config']['question_hashes'] = question_hashes
    persona_dict['config']['answer_index'] = answer_index
    persona_dict['config']['model'] = "gpt_41_mini"
    persona_dict['answers'] = persona_answers
    
    sjt_answers.append(persona_dict)